# 06c · Ana model C (IR-telafili hibrit) — tüm tablolar ve İngilizce şekiller
Ön koşul: 02, 03, 03b, 04, 05, **05c, 03c, 04b** tamamlanmış olmalı. Eğitim yok.

| Çıktı | Yer |
|---|---|
| Tek Excel (T1–T15 + ham satırlar) | `results\excel\V26C_ozet.xlsx` |
| Şekiller (PDF vektör + PNG 600 dpi, JESTECH sütun ölçüleri) | `graphs\C\C_Fig01 … C_Fig12` |

Adım 1 (isteğe bağlı, GPU, ~5 dk): RC uzantısının en iyi seed'i için η serileri (Şekil 9) ve Proposed-S verimliliği (Şekil 12) — yoksa üretilir.
Yeni modül: `models\c_report.py` (yalnız pandas/matplotlib; mevcut `figures.py`/`report.py` değişmez).

In [ ]:
import sys, os, json, time, warnings
import os; ROOT = os.path.abspath('..')  # repository root (run notebooks from the notebooks/ folder)
SMOKE = False
sys.path.insert(0, ROOT)
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, torch
from data_utils import *
from models.experiment import *
from models import engine
exp = Experiment(ROOT, smoke=SMOKE)
exp.load_data()
print('Cihaz:', exp.dev, '| W =', exp.cfg['W'])

In [ ]:
# Adım 1 — eksik ara çıktılar (eğitim yok). İkinci çalıştırmada atlanır.
RUN_EXTRA = True
if RUN_EXTRA:
    rr = [r for r in exp.records('ablation_b') if r['name'] == 'RC + akım girişi']
    if rr:
        b = min(rr, key=lambda r: r['val_rmse_final'])
        if not table_exists(exp.paths('ablation_b', b['name'], b['seed'])[2]):
            exp.run(b['name'], b['spec'], b['seed'], 'ablation_b', series=True, ablations=RC_ABLATIONS_TEST)
        print('✅ RC uzantısı serisi: seed', b['seed'])
    f = exp.p['configs'] / 'verimlilik_cs.json'
    NAME_S = 'C: eşit boyut (h=64)'
    if not f.exists() and (exp.p['runs'] / 'ablation_c').exists():
        m, rec = exp.load_model('ablation_c', NAME_S, exp.cfg['FINAL_SEEDS'][0])
        X = next(iter(exp.datasets(rec['spec']['window'])['test'].values())).X[:rec['spec']['window']].unsqueeze(0)
        e = engine.efficiency(m, X, n_rep=200)
        rr = [r for r in exp.records('ablation_c') if r['name'] == NAME_S]
        e.update({'model': NAME_S, 'W (s)': rec['spec']['window'], 'eğitim süresi ort (dk)': np.mean([r['train']['train_time_s'] for r in rr]) / 60,
                  'epoch süresi ort (s)': np.mean([r['train']['epoch_time_mean_s'] for r in rr]),
                  'eğitim GPU bellek tepe (GB)': np.mean([r['train'].get('train_peak_mem_gb', np.nan) for r in rr]), 'ön işleme (ms / örnek)': exp.pre_ms})
        pd.DataFrame([e]).to_json(f, orient='records', force_ascii=False, indent=1)
        print('✅ Proposed-S verimlilik:', {k: (round(v, 3) if isinstance(v, float) else v) for k, v in e.items()})

In [ ]:
# Adım 2 — veriyi topla ve tabloları üret
from models import c_report as R
import matplotlib.pyplot as plt
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 40)
D = R.gather(exp)
T = R.tables(D)
out = R.write_excel(T, exp.p['excel'] / 'V26C_ozet.xlsx')
print('✅', out, '\n')
print('T1 — ana tablo:'); print(T['T1 Ana tablo (makale)'].drop(columns='model').round(3).to_string(index=False))
print('\nT5 — anlamlılık (C vs):'); print(T['T5 Anlamlılık (C vs)'].drop(columns='karşılaştırma').round(4).to_string(index=False))

In [ ]:
print('T6 — faktöriyel (görülen / −20 °C · val):'); print(T['T6b Faktöriyel metin'].to_string(index=False))
print('\nT7 — ablasyon + duyarlılık:')
cols = ['varyant', 'n_seed', 'val (%)', 'RMSE görülen', 'RMSE −20 °C', 'sapma −20 °C', 'C daha iyi (−20)', 'p Wilcoxon (−20)', 'C daha iyi (gör)', 'p Wilcoxon (gör)']
print(T['T7 Ablasyon + duyarlılık'].reindex(columns=cols).round(4).to_string(index=False))
print('\nT10 — basamak direnci (veri) ↔ öğrenilen (ölçülen hücre sıcaklığında):')
print(T['T10 Basamak direnci vs öğrenilen'].round(2).to_string(index=False)); print(T['T10b Sıcaklık duyarlılığı'].round(3).to_string(index=False))

In [ ]:
# Adım 3 — şekiller (İngilizce) → graphs\C\
R.DPI = 600; R.FORMATS = ('pdf', 'png'); R.style()
G = exp.p['graphs'] / 'C'
for f in R.FIGS:
    try:
        fig = f(D, G)
        if fig is not None: plt.show()
        print('✅', f.__name__)
    except Exception as e:
        print('⚠️', f.__name__, '→', repr(e))
print('Şekiller →', G)

### Şekil listesi
| Dosya | İçerik | Ana metin önerisi |
|---|---|---|
| C_Fig01 | RMSE, sıcaklık bazında (görülen / görülmemiş) | ✔ |
| C_Fig02 | −20 °C, çevrim bazında | ek |
| C_Fig03 | SoC takibi ve hata (4 durum) | ✔ |
| C_Fig04 | Ölçülen gerilim ↔ IR-telafili gerilim (sıcaklıklar çakışır) | ✔ |
| C_Fig05 | Ablasyon + duyarlılık (faktöriyel varyantlar) | ✔ |
| C_Fig06 | Öğrenilen R0(T) ↔ veriden basamak direnci ↔ ECM | ✔ |
| C_Fig07 | Hata – SoC (−10 / −20 °C): soğuk sapma mekanizması | ✔ |
| C_Fig08 | Sağlamlık | ek (veya tablo) |
| C_Fig09 | Çok ölçekli RC uzantısı (η ayrıştırması + R/τ) | ek / tartışma |
| C_Fig10–12 | Sızıntı, öğrenme eğrileri, doğruluk–boyut | ek |